# 1. Limpieza y preparación de datos

**Objetivo:** leer el archivo `datos/datalog.txt`, extraer las mediciones de temperatura y humedad de los sensores Casa Guadua, Casa Plástico y Ambiente, limpiar valores erróneos y asignar un tiempo relativo en minutos.

In [4]:
from pathlib import Path

import re
import pandas as pd
import numpy as np

ruta_raw = Path('DATALOG.TXT')

## 1.2 Leer el archivo raw


In [5]:
with ruta_raw.open('r', encoding='utf-8') as f:
    lineas = f.readlines()

## 1.3 Parsear las líneas para extraer cada muestra.


In [6]:
muestras = []
muestra_actual = {}
for linea in lineas:
    linea = linea.strip()
    if not linea or linea.startswith('===') or linea.startswith('Formato:'):
        continue
    if linea.startswith("Muestra #"):
        if muestra_actual:
            muestras.append(muestra_actual)
            muestra_actual = {}
        num = int(re.search(r"\d+", linea).group())
        muestra_actual["muestra"] = num
    elif linea.startswith("S1 Casa Guadua:"):
        valores = re.findall(r'(-?\d+(?:\.\d+)?)', linea.split(':', 1)[1])
        if len(valores) >= 2:
            muestra_actual["T_guadua"] = float(valores[0])
            muestra_actual["H_guadua"] = float(valores[1])
    elif linea.startswith("S2 Casa Plastico:"):
        valores = re.findall(r'(-?\d+(?:\.\d+)?)', linea.split(':', 1)[1])
        if len(valores) >= 2:
            muestra_actual["T_plastico"] = float(valores[0])
            muestra_actual["H_plastico"] = float(valores[1])
    elif linea.startswith("S4 Ambiente :"):
        valores = re.findall(r'(-?\d+(?:\.\d+)?)', linea.split(':', 1)[1])
        if len(valores) >= 2:
            muestra_actual["T_ambiente"] = float(valores[0])
            muestra_actual["H_ambiente"] = float(valores[1])
    elif linea.startswith("--------------------------------"):
        if muestra_actual:
            muestras.append(muestra_actual)
            muestra_actual = {}

if muestra_actual:
    muestras.append(muestra_actual)

## 1.4 Crear DataFrame y limpiar


In [7]:
df = pd.DataFrame(muestras)
df_clean = df.replace(-999, np.nan).dropna().copy()
df_clean['tiempo_min'] = df_clean['muestra'] - 1
columnas = ['muestra', 'tiempo_min', 'T_guadua', 'H_guadua', 'T_plastico', 'H_plastico', 'T_ambiente', 'H_ambiente']
df_clean = df_clean[columnas].reset_index(drop=True)

## 1.5 Guardar datos limpios


In [8]:
ruta_salida = Path('datos_limpios.csv')
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(ruta_salida, index=False)
print(f'Se procesaron {len(df_clean)} muestras válidas.')
df_clean.head()

Se procesaron 9931 muestras válidas.


,muestra,tiempo_min,T_guadua,H_guadua,T_plastico,H_plastico,T_ambiente,H_ambiente
0,1,0,0.6,0.0,0.8,55.0,-2.4,10.0
1,2,1,24.1,55.0,25.5,60.0,20.1,63.0
2,3,2,24.3,55.0,25.9,60.0,20.1,63.0
3,4,3,24.2,55.0,25.1,60.0,20.1,63.0
4,5,4,24.4,55.0,25.2,60.0,20.0,63.0


In [9]:
df_clean[['muestra', 'tiempo_min', 'T_guadua', 'T_plastico', 'T_ambiente']].tail()

,muestra,tiempo_min,T_guadua,T_plastico,T_ambiente
9926,9927,9926,23.8,24.8,20.2
9927,9928,9927,23.8,24.2,21.9
9928,9929,9928,23.7,25.2,21.4
9929,9930,9929,23.1,25.6,21.7
9930,9931,9930,24.3,26.4,21.1
